# 03_langgraph_agent_test.ipynb
이 노트북은 새로운 라우터 중심 아키텍처(@design.md)가 적용된 서사 에이전트를 테스트합니다.

### 주요 테스트 항목:
1. **초기 설정 및 데이터 로드 (`history_node`)**
2. **로그라인 월드빌딩 생성 (`generator_node`)**
3. **의도 기반 라우팅 (`router_logic`)**
4. **워크플로우 검증 (`response_check_node`)**

In [ ]:
import os
import sys
from pathlib import Path

# 프로젝트 루트 경로 설정 (노트북 위치 기준 상위 폴더)
ROOT_PATH = os.path.abspath("..")
if ROOT_PATH not in sys.path:
    sys.path.append(ROOT_PATH)

os.chdir(ROOT_PATH)
print(f"Current Working Directory: {os.getcwd()}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from src.narrative_agent import build_narrative_graph
import json

### 1. 에이전트 그래프 빌드 및 시각화
`src/narrative_agent.py`에서 정의된 새로운 그래프를 생성합니다.

In [ ]:
graph = build_narrative_graph()
print("✅ Narrative Graph Compiled Successfully.")

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization requires 'pygraphviz' or 'mermaid' tools. Skipping...")

### 2. 사용 사례 1: 초기 로그라인 생성 (Logline Building)
사용자가 아이디어를 던졌을 때, `history -> generator -> check` 흐름을 시뮬레이션합니다.

*참고: OPENAI_API_KEY가 설정되어 있어야 generator가 작동합니다.*

In [ ]:
# 초기 질문
user_input = "기억을 잃은 요리사가 음식을 통해 과거의 맛을 찾아가는 이야기를 만들고 싶어."

initial_state = {
    "messages": [HumanMessage(content=user_input)],
    "user_data": {}, # history_node에서 로드됨
    "workflow": {}, # history_node에서 로드됨
    "next_node": "history",
    "last_response": None,
    "validation_results": {}
}

print("--- Starting Test Case 1 ---")
try:
    # 스트리밍 실행으로 노드별 흐름 관찰
    for chunk in graph.stream(initial_state, {"configurable": {"thread_id": "case1"}}):
        for node, output in chunk.items():
            print(f"\n[Node: {node}]")
            if "last_response" in output and output["last_response"]:
                print(f"Agent Response: {output['last_response']}")
            if "validation_results" in output:
                print(f"Validation: {output['validation_results']}")
except Exception as e:
    print(f"Error during execution: {e}")

### 3. 사용 사례 2: 의도 기반 라우팅 테스트 (Router Mocking)
사용자가 데이터를 저장해달라고 요청했을 때, 라우터가 `update` 노드로 올바르게 분기하는지 확인합니다.

In [ ]:
mock_state = {
    "messages": [HumanMessage(content="지금 내용을 캐릭터 파일에 저장해줘.")],
    "next_node": "update", # 의도 분석 결과로 가정
    "user_data": {}, 
    "workflow": {}
}

print("--- Starting Test Case 2 (Routing to Update) ---")
# update 노드 직접 실행 테스트
from src.narrative_agent import router_logic
next_step = router_logic(mock_state)
print(f"Router Verdict: {next_step} (Expected: update)")

### 4. 사용 사례 3: 검증 피드백 및 재생성 (Validation Loop)
생성된 응답에 필수 요구사항(예: 주인공 정보)이 누락되었을 때, `response_check` 노드가 어떻게 반응하는지 테스트합니다.

In [ ]:
from src.narrative_agent import response_check_node

invalid_response_state = {
    "last_response": "이 이야기는 아주 재미있을 거예요!", # 주인공 이름이 누락된 응답
    "workflow": {
        "output_requirements": ["주인공"]
    }
}

print("--- Starting Test Case 3 (Validation Failure) ---")
check_result = response_check_node(invalid_response_state)
print(f"Validation Result: {check_result['validation_results']}")
print(f"Next Node Decision: {check_result['next_node']} (Expected: router or generator)")